In [1]:
import sys

print("Python:", sys.version)
print("✅ StudyMate final project started")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
✅ StudyMate final project started


In [2]:
!pip -q install sentence-transformers faiss-cpu streamlit fastapi uvicorn requests feedparser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 66.6 MB/s eta 0:00:00


In [3]:
import os
import re
import json
import requests
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


In [4]:
dataset = """
STUDYMATE ACADEMIC KNOWLEDGE BASE

1. LARGE LANGUAGE MODELS

Large Language Models, commonly known as LLMs, are artificial intelligence
models trained on very large collections of text. They can understand and
generate human-like language.

LLMs can be used for question answering, summarization, translation,
content generation, programming assistance, and educational applications.

Examples of language model families include T5, FLAN-T5, Llama, Mistral,
Gemma, Qwen, and other transformer-based models.

2. LIMITATIONS OF LANGUAGE MODELS

Language models may have incomplete or outdated knowledge. They may also
not have access to private or organization-specific documents.

Because of this, an LLM may produce an answer that is not based on the
latest or most relevant information.

Retrieval-Augmented Generation can help address this limitation by
connecting a language model with an external knowledge source.

3. RETRIEVAL-AUGMENTED GENERATION

Retrieval-Augmented Generation, commonly called RAG, is an AI technique
that combines information retrieval with language generation.

Instead of asking a language model to answer a question only from its
internal knowledge, a RAG system first searches a collection of documents
for relevant information.

The retrieved information is then provided to the language model as
context. The language model uses this context to generate the final answer.

4. RAG PIPELINE

A typical RAG pipeline contains the following stages:

Dataset collection
Document loading
Text preprocessing
Text chunking
Embedding generation
Vector database storage
Similarity search
Document retrieval
Prompt construction
Language model generation
Final answer generation

5. DOCUMENT CHUNKING

Chunking means dividing a large document into smaller pieces called chunks.

Chunking is important because language models have limits on the amount of
text they can process at one time.

Good chunks should contain meaningful information and should not be too
large or too small.

Chunk overlap can be used so that information near the boundary of two
chunks is not completely lost.

6. TEXT EMBEDDINGS

An embedding is a numerical representation of text.

Texts with similar meanings should have similar vector representations.

Embeddings allow a computer to compare the semantic meaning of a user
question with stored document chunks.

The all-MiniLM-L6-v2 model is a popular sentence-transformer model and
produces 384-dimensional embeddings.

7. VECTOR DATABASE

A vector database stores numerical representations of documents and allows
efficient similarity searching.

FAISS is a library developed for efficient similarity search and clustering
of dense vectors.

In StudyMate, FAISS is used as the vector database.

8. RETRIEVAL

When a user asks a question, the question is converted into an embedding.

The question embedding is compared with the document embeddings stored in
FAISS.

The most similar document chunks are retrieved.

These retrieved chunks are then supplied to the language model as context.

9. PROMPT ENGINEERING

Prompt engineering means designing instructions given to a language model.

A RAG prompt normally contains the retrieved context, the user's question,
and instructions explaining how the model should answer.

StudyMate instructs the model to answer using the retrieved academic context.

10. FLAN-T5

FLAN-T5 is a sequence-to-sequence language model developed by Google.

It can perform tasks such as question answering, summarization, translation,
and text generation.

StudyMate uses FLAN-T5-small because it is relatively lightweight and can
run in a Google Colab environment.

11. ADVANTAGES OF RAG

RAG allows a language model to use information from an external knowledge
base.

It can reduce dependence on the model's internal knowledge.

RAG also makes it possible to update the knowledge base without retraining
the complete language model.

Retrieved sources can be displayed to users, making the application more
transparent.

12. STUDYMATE APPLICATION

StudyMate is an academic assistant designed for students.

Students can ask questions about Large Language Models and
Retrieval-Augmented Generation.

The application searches its academic knowledge base, retrieves relevant
information, and generates a concise answer.

StudyMate also displays the retrieved source documents.

13. EXTERNAL RESEARCH APIS

StudyMate uses external APIs to provide additional research information.

The Wikipedia API provides general topic summaries.

The arXiv API provides access to research papers.

The OpenAlex API provides scholarly publication information.

The Crossref API provides research publication metadata and DOI information.

The Open Library API provides information about books and authors.

14. API INTEGRATION

Application programming interfaces, or APIs, allow software applications
to communicate with external services.

StudyMate uses five external APIs to extend its research capabilities.

The application also exposes its own REST API so that other applications
can send questions to StudyMate programmatically.

15. FASTAPI

FastAPI is a Python framework for building web APIs.

StudyMate exposes an API endpoint called /ask.

A client can send a question to the endpoint and receive a JSON response
containing the generated answer and retrieved sources.

16. STUDENT USE CASE

A student studying artificial intelligence can ask StudyMate questions such
as:

What is Retrieval-Augmented Generation?

What is an embedding?

Why is FAISS used in RAG?

What are the advantages of RAG?

What is prompt engineering?

The system retrieves relevant academic information before generating the
answer.

17. CONCLUSION

StudyMate demonstrates a complete end-to-end Retrieval-Augmented Generation
application.

The project includes data creation, preprocessing, chunking, embeddings,
FAISS vector search, retrieval, prompt engineering, language generation,
external API integration, Streamlit deployment, and a FastAPI REST API.
"""

with open("/content/academic_notes.txt", "w", encoding="utf-8") as f:
    f.write(dataset)

print("✅ Academic dataset created")
print("📄 File: /content/academic_notes.txt")
print("📊 Characters:", len(dataset))

✅ Academic dataset created
📄 File: /content/academic_notes.txt
📊 Characters: 6008


In [5]:
with open("/content/academic_notes.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("✅ Dataset loaded successfully")
print("Characters:", len(text))
print("\nPreview:\n")
print(text[:1500])

✅ Dataset loaded successfully
Characters: 6008

Preview:


STUDYMATE ACADEMIC KNOWLEDGE BASE

1. LARGE LANGUAGE MODELS

Large Language Models, commonly known as LLMs, are artificial intelligence
models trained on very large collections of text. They can understand and
generate human-like language.

LLMs can be used for question answering, summarization, translation,
content generation, programming assistance, and educational applications.

Examples of language model families include T5, FLAN-T5, Llama, Mistral,
Gemma, Qwen, and other transformer-based models.

2. LIMITATIONS OF LANGUAGE MODELS

Language models may have incomplete or outdated knowledge. They may also
not have access to private or organization-specific documents.

Because of this, an LLM may produce an answer that is not based on the
latest or most relevant information.

Retrieval-Augmented Generation can help address this limitation by
connecting a language model with an external knowledge source.

3. RETRIEVAL-AUGMENTE

In [6]:
import re

def clean_text(text):
    # Remove carriage returns
    text = text.replace("\r", "\n")

    # Remove extra spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


cleaned_text = clean_text(text)

print("✅ Data cleaning completed")
print("Original characters:", len(text))
print("Cleaned characters:", len(cleaned_text))

✅ Data cleaning completed
Original characters: 6008
Cleaned characters: 6006


In [7]:
paragraphs = [
    p.strip()
    for p in cleaned_text.split("\n\n")
    if p.strip()
]

chunks = []

current_chunk = ""

for paragraph in paragraphs:

    # Keep chunks reasonably sized
    if len(current_chunk) + len(paragraph) <= 900:
        current_chunk += paragraph + "\n\n"

    else:
        if current_chunk.strip():
            chunks.append(current_chunk.strip())

        current_chunk = paragraph + "\n\n"

# Add final chunk
if current_chunk.strip():
    chunks.append(current_chunk.strip())


print("✅ Chunking completed")
print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks[:5], 1):
    print("\n" + "=" * 60)
    print("CHUNK", i)
    print(chunk[:600])

✅ Chunking completed
Number of chunks: 8

CHUNK 1
STUDYMATE ACADEMIC KNOWLEDGE BASE

1. LARGE LANGUAGE MODELS

Large Language Models, commonly known as LLMs, are artificial intelligence
models trained on very large collections of text. They can understand and
generate human-like language.

LLMs can be used for question answering, summarization, translation,
content generation, programming assistance, and educational applications.

Examples of language model families include T5, FLAN-T5, Llama, Mistral,
Gemma, Qwen, and other transformer-based models.

2. LIMITATIONS OF LANGUAGE MODELS

Language models may have incomplete or outdated knowledge

CHUNK 2
Retrieval-Augmented Generation can help address this limitation by
connecting a language model with an external knowledge source.

3. RETRIEVAL-AUGMENTED GENERATION

Retrieval-Augmented Generation, commonly called RAG, is an AI technique
that combines information retrieval with language generation.

Instead of asking a language model to a

In [8]:
print("🔄 Loading embedding model...")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("✅ Embedding model loaded")

🔄 Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded


In [9]:
print("🔄 Generating embeddings...")

embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")

print("\n✅ Embeddings generated")
print("Embedding shape:", embeddings.shape)

🔄 Generating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Embeddings generated
Embedding shape: (8, 384)


In [10]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("✅ FAISS vector database created")
print("Vector dimension:", dimension)
print("Number of stored vectors:", index.ntotal)

✅ FAISS vector database created
Vector dimension: 384
Number of stored vectors: 8


In [11]:
def retrieve_documents(query, top_k=3):

    # Convert the user's question into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Search FAISS
    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx >= 0:
            results.append({
                "score": float(score),
                "text": chunks[idx]
            })

    return results


print("✅ Retriever function created")

✅ Retriever function created


In [12]:
question = "What is Retrieval-Augmented Generation?"

results = retrieve_documents(
    question,
    top_k=3
)

print("QUESTION:")
print(question)

for i, result in enumerate(results, 1):

    print("\n" + "=" * 70)

    print(
        f"SOURCE {i} | "
        f"Similarity: {result['score']:.4f}"
    )

    print(result["text"])

QUESTION:
What is Retrieval-Augmented Generation?

SOURCE 1 | Similarity: 0.6756
Retrieval-Augmented Generation can help address this limitation by
connecting a language model with an external knowledge source.

3. RETRIEVAL-AUGMENTED GENERATION

Retrieval-Augmented Generation, commonly called RAG, is an AI technique
that combines information retrieval with language generation.

Instead of asking a language model to answer a question only from its
internal knowledge, a RAG system first searches a collection of documents
for relevant information.

The retrieved information is then provided to the language model as
context. The language model uses this context to generate the final answer.

4. RAG PIPELINE

A typical RAG pipeline contains the following stages:

SOURCE 2 | Similarity: 0.3937
StudyMate uses five external APIs to extend its research capabilities.

The application also exposes its own REST API so that other applications
can send questions to StudyMate programmatically.

15. 

In [13]:
print("🔄 Loading FLAN-T5-small...")

tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-small"
)

llm_model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-small"
)

print("✅ FLAN-T5-small loaded")

🔄 Loading FLAN-T5-small...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ FLAN-T5-small loaded


In [14]:
def ask_rag(question, top_k=3):

    # 1. Retrieve relevant documents
    results = retrieve_documents(
        question,
        top_k=top_k
    )

    # 2. Combine retrieved chunks
    context = "\n\n".join(
        result["text"]
        for result in results
    )

    # 3. Create RAG prompt
    prompt = f"""
You are StudyMate, an academic assistant.

Answer the question using only the information
provided in the context.

Give a short, clear and accurate answer.

Context:
{context}

Question:
{question}

Answer:
"""

    # 4. Tokenize prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # 5. Generate answer
    outputs = llm_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=100,
        num_beams=4,
        do_sample=False
    )

    # 6. Decode answer
    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, results


print("✅ Complete RAG pipeline created")

✅ Complete RAG pipeline created


In [15]:
question = "What is Retrieval-Augmented Generation?"

answer, sources = ask_rag(question)

print("=" * 70)
print("🎓 STUDYMATE RAG TEST")
print("=" * 70)

print("\nQUESTION:")
print(question)

print("\nANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")

for i, source in enumerate(sources, 1):

    print(
        f"\nSource {i} "
        f"| Similarity: "
        f"{source['score']:.4f}"
    )

    print(source["text"][:500])

🎓 STUDYMATE RAG TEST

QUESTION:
What is Retrieval-Augmented Generation?

ANSWER:
RETRIEVAL-AUGMENTED GENERATION Retrieval-Augmented Generation, commonly called RAG, is an AI technique that combines information retrieval with language generation. Instead of asking a language model to answer a question only from its internal knowledge, a RAG system first searches a collection of documents for relevant information. The retrieved information is then provided to the language model as context. The language model uses this context to generate the

RETRIEVED SOURCES:

Source 1 | Similarity: 0.6756
Retrieval-Augmented Generation can help address this limitation by
connecting a language model with an external knowledge source.

3. RETRIEVAL-AUGMENTED GENERATION

Retrieval-Augmented Generation, commonly called RAG, is an AI technique
that combines information retrieval with language generation.

Instead of asking a language model to answer a question only from its
internal knowledge, a RAG system

In [18]:
# CELL 16 — FIXED WIKIPEDIA API

def wikipedia_api(topic):

    # Set a User-Agent to comply with Wikipedia API guidelines
    headers = {
        "User-Agent": "StudyMate (colab-studymate-project@example.com)"
    }

    # First search for the correct Wikipedia page
    search_url = "https://en.wikipedia.org/w/api.php"

    search_params = {
        "action": "query",
        "list": "search",
        "srsearch": topic,
        "format": "json",
        "utf8": 1
    }

    search_response = requests.get(
        search_url,
        params=search_params,
        headers=headers,
        timeout=15
    )

    search_response.raise_for_status()

    search_data = search_response.json()

    search_results = search_data.get(
        "query", {}
    ).get(
        "search", []
    )

    if not search_results:
        return {
            "title": None,
            "summary": "No Wikipedia result found.",
            "url": None
        }

    # Get the best matching page title
    page_title = search_results[0]["title"]

    # Get page summary
    summary_url = (
        "https://en.wikipedia.org/api/rest_v1/page/summary/"
        + requests.utils.quote(page_title)
    )

    summary_response = requests.get(
        summary_url,
        headers=headers,
        timeout=15
    )

    if summary_response.status_code == 200:

        data = summary_response.json()

        return {
            "title": data.get("title"),
            "summary": data.get("extract"),
            "url": (
                data.get("content_urls", {})
                .get("desktop", {})
                .get("page")
            )
        }

    return {
        "title": page_title,
        "summary": "Wikipedia page found, but summary could not be loaded.",
        "url": (
            "https://en.wikipedia.org/wiki/"
            + requests.utils.quote(
                page_title.replace(" ", "_")
            )
        )
    }


# Test the API
wiki_result = wikipedia_api(
    "Retrieval Augmented Generation"
)

print("🌐 WIKIPEDIA API")
print("=" * 60)

print("Title:", wiki_result["title"])

print("\nSummary:")
print(wiki_result["summary"])

print("\nURL:")
print(wiki_result["url"])

🌐 WIKIPEDIA API
Title: Retrieval-augmented generation

Summary:
Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM's pre-existing training data. This allows LLMs to use domain-specific and/or updated information that is not available in the training data. For example, this enables LLM-based chatbots to access internal company data or generate responses based on authoritative sources. The technique was first proposed in 2020 and has since become a widely adopted approach in modern AI systems.

URL:
https://en.wikipedia.org/wiki/Retrieval-augmented_generation


In [20]:
# CELL 17 — arXiv API

import feedparser

def arxiv_api(query, limit=3):

    url = "https://export.arxiv.org/api/query"

    params = {
        "search_query": f"all:{query}",
        "start": 0,
        "max_results": limit
    }

    headers = {
        "User-Agent": "StudyMateAcademicAssistant/1.0"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=30
    )

    print("arXiv status:", response.status_code)

    response.raise_for_status()

    feed = feedparser.parse(response.text)

    papers = []

    for entry in feed.entries:

        papers.append({
            "title": entry.get("title", "").strip(),
            "summary": entry.get("summary", "").strip(),
            "url": entry.get("link", "")
        })

    return papers


papers = arxiv_api(
    "retrieval augmented generation",
    3
)

print("\n📄 arXiv API")
print("=" * 60)

for i, paper in enumerate(papers, 1):

    print(f"\nPaper {i}")
    print("Title:", paper["title"])
    print("URL:", paper["url"])

arXiv status: 200

📄 arXiv API

Paper 1
Title: AR-RAG: Autoregressive Retrieval Augmentation for Image Generation
URL: https://arxiv.org/abs/2506.06962v3

Paper 2
Title: Intelligent Interaction Strategies for Context-Aware Cognitive Augmentation
URL: https://arxiv.org/abs/2504.13684v1

Paper 3
Title: Factually: Exploring Wearable Fact-Checking for Augmented Truth Discernment
URL: https://arxiv.org/abs/2504.17204v1


In [21]:
# CELL 18 — OpenAlex API

def openalex_api(query, limit=3):

    url = "https://api.openalex.org/works"

    params = {
        "search": query,
        "per-page": limit
    }

    headers = {
        "User-Agent": "StudyMateAcademicAssistant/1.0"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=30
    )

    print("OpenAlex status:", response.status_code)

    response.raise_for_status()

    data = response.json()

    results = []

    for work in data.get("results", []):

        results.append({
            "title": work.get("title"),
            "year": work.get("publication_year"),
            "doi": work.get("doi"),
            "url": work.get("id")
        })

    return results


openalex_results = openalex_api(
    "retrieval augmented generation",
    3
)

print("\n🔬 OPENALEX API")
print("=" * 60)

for i, item in enumerate(openalex_results, 1):

    print(f"\nResult {i}")
    print("Title:", item["title"])
    print("Year:", item["year"])
    print("DOI:", item["doi"])
    print("OpenAlex:", item["url"])

OpenAlex status: 200

🔬 OPENALEX API

Result 1
Title: Retrieval-Augmented Generation for Large Language Models: A Survey
Year: 2023
DOI: https://doi.org/10.48550/arxiv.2312.10997
OpenAlex: https://openalex.org/W4389984066

Result 2
Title: Active Retrieval Augmented Generation
Year: 2023
DOI: https://doi.org/10.18653/v1/2023.emnlp-main.495
OpenAlex: https://openalex.org/W4389519118

Result 3
Title: Benchmarking Large Language Models in Retrieval-Augmented Generation
Year: 2024
DOI: https://doi.org/10.1609/aaai.v38i16.29728
OpenAlex: https://openalex.org/W4393147129


In [22]:
# CELL 19 — Crossref API

def crossref_api(query, limit=3):

    url = "https://api.crossref.org/works"

    params = {
        "query": query,
        "rows": limit
    }

    headers = {
        "User-Agent": "StudyMateAcademicAssistant/1.0"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=30
    )

    print("Crossref status:", response.status_code)

    response.raise_for_status()

    data = response.json()

    results = []

    for item in data.get("message", {}).get("items", []):

        title_list = item.get("title", [""])

        title = title_list[0] if title_list else ""

        published = item.get("published", {})
        date_parts = published.get("date-parts", [[None]])

        year = date_parts[0][0] if date_parts and date_parts[0] else None

        results.append({
            "title": title,
            "doi": item.get("DOI"),
            "publisher": item.get("publisher"),
            "year": year
        })

    return results


crossref_results = crossref_api(
    "retrieval augmented generation",
    3
)

print("\n🔗 CROSSREF API")
print("=" * 60)

for i, item in enumerate(crossref_results, 1):

    print(f"\nResult {i}")
    print("Title:", item["title"])
    print("DOI:", item["doi"])
    print("Publisher:", item["publisher"])
    print("Year:", item["year"])

Crossref status: 200

🔗 CROSSREF API

Result 1
Title: Retrieval‐Augmented Generation
DOI: 10.1002/9781394374717.ch03
Publisher: Wiley
Year: 2026

Result 2
Title: Graph‐based Retrieval‐augmented Generation
DOI: 10.1002/9781394374717.ch07
Publisher: Wiley
Year: 2026

Result 3
Title: Efficient Information Retrieval and Response Generation with Retrieval-Augmented Generation (RAG)
DOI: 10.59350/q2pq3-0fv85
Publisher: Front Matter
Year: 2024


In [23]:
# CELL 20 — Open Library API

def openlibrary_api(query, limit=3):

    url = "https://openlibrary.org/search.json"

    params = {
        "q": query,
        "limit": limit
    }

    headers = {
        "User-Agent": "StudyMateAcademicAssistant/1.0"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=30
    )

    print("Open Library status:", response.status_code)

    response.raise_for_status()

    data = response.json()

    books = []

    for book in data.get("docs", []):

        authors = book.get(
            "author_name",
            ["Unknown"]
        )

        books.append({
            "title": book.get("title", "Unknown"),
            "author": authors[0] if authors else "Unknown",
            "year": book.get("first_publish_year")
        })

    return books


books = openlibrary_api(
    "large language models",
    3
)

print("\n📚 OPEN LIBRARY API")
print("=" * 60)

for i, book in enumerate(books, 1):

    print(f"\nBook {i}")
    print("Title:", book["title"])
    print("Author:", book["author"])
    print("Year:", book["year"])

Open Library status: 200

📚 OPEN LIBRARY API

Book 1
Title: Build a Large Language Model (from Scratch)
Author: Sebastian Raschka
Year: 2024

Book 2
Title: Hands-On Large Language Models
Author: Jay Alammar
Year: 2024

Book 3
Title: Large Language Models
Author: Oswald Campesato
Year: 2024


In [24]:
# CELL 21 — Verify all 5 APIs

print("=" * 70)
print("🎓 STUDYMATE — EXTERNAL API VERIFICATION")
print("=" * 70)

# API 1
print("\n1️⃣ Wikipedia API")
if wiki_result.get("title"):
    print("✅ WORKING")
    print("   Result:", wiki_result["title"])
else:
    print("❌ FAILED")

# API 2
print("\n2️⃣ arXiv API")
if papers:
    print("✅ WORKING")
    print("   Papers found:", len(papers))
else:
    print("❌ FAILED")

# API 3
print("\n3️⃣ OpenAlex API")
if openalex_results:
    print("✅ WORKING")
    print("   Results found:", len(openalex_results))
else:
    print("❌ FAILED")

# API 4
print("\n4️⃣ Crossref API")
if crossref_results:
    print("✅ WORKING")
    print("   Results found:", len(crossref_results))
else:
    print("❌ FAILED")

# API 5
print("\n5️⃣ Open Library API")
if books:
    print("✅ WORKING")
    print("   Books found:", len(books))
else:
    print("❌ FAILED")

print("\n" + "=" * 70)
print("🎉 STUDYMATE API TEST COMPLETE")
print("=" * 70)

🎓 STUDYMATE — EXTERNAL API VERIFICATION

1️⃣ Wikipedia API
✅ WORKING
   Result: Retrieval-augmented generation

2️⃣ arXiv API
✅ WORKING
   Papers found: 3

3️⃣ OpenAlex API
✅ WORKING
   Results found: 3

4️⃣ Crossref API
✅ WORKING
   Results found: 3

5️⃣ Open Library API
✅ WORKING
   Books found: 3

🎉 STUDYMATE API TEST COMPLETE


In [25]:
!pip -q install fastapi uvicorn nest-asyncio

In [26]:
# CELL 23 — STUDYMATE OWN REST API

from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional

app = FastAPI(
    title="StudyMate API",
    description="Academic RAG Assistant API",
    version="1.0"
)


class QuestionRequest(BaseModel):
    question: str
    top_k: Optional[int] = 3


@app.get("/")
def home():
    return {
        "application": "StudyMate",
        "status": "running",
        "description": "Academic RAG Assistant API"
    }


@app.post("/ask")
def ask_question(request: QuestionRequest):

    answer, sources = ask_rag(
        request.question,
        top_k=request.top_k
    )

    source_data = []

    for source in sources:
        source_data.append({
            "similarity": round(
                source["score"],
                4
            ),
            "text": source["text"]
        })

    return {
        "question": request.question,
        "answer": answer,
        "sources": source_data
    }


print("✅ StudyMate FastAPI application created")
print("📌 Endpoint: POST /ask")

✅ StudyMate FastAPI application created
📌 Endpoint: POST /ask


In [27]:
# CELL 24 — START FASTAPI SERVER

import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()


def run_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="warning"
    )


api_thread = threading.Thread(
    target=run_api,
    daemon=True
)

api_thread.start()

print("🚀 StudyMate API server started")
print("🌐 Port: 8000")

🚀 StudyMate API server started
🌐 Port: 8000


In [28]:
# CELL 25 — TEST STUDYMATE API

import time
import requests

time.sleep(2)

api_url = "http://127.0.0.1:8000/ask"

payload = {
    "question": "What is Retrieval-Augmented Generation?",
    "top_k": 3
}

response = requests.post(
    api_url,
    json=payload,
    timeout=60
)

print("Status code:", response.status_code)

data = response.json()

print("\n🎓 STUDYMATE OWN API")
print("=" * 60)

print("\nQuestion:")
print(data["question"])

print("\nAnswer:")
print(data["answer"])

print("\nSources:")

for i, source in enumerate(
    data["sources"],
    1
):
    print(
        f"\nSource {i} "
        f"| Similarity: {source['similarity']}"
    )
    print(source["text"][:300])

Status code: 200

🎓 STUDYMATE OWN API

Question:
What is Retrieval-Augmented Generation?

Answer:
RETRIEVAL-AUGMENTED GENERATION Retrieval-Augmented Generation, commonly called RAG, is an AI technique that combines information retrieval with language generation. Instead of asking a language model to answer a question only from its internal knowledge, a RAG system first searches a collection of documents for relevant information. The retrieved information is then provided to the language model as context. The language model uses this context to generate the

Sources:

Source 1 | Similarity: 0.6756
Retrieval-Augmented Generation can help address this limitation by
connecting a language model with an external knowledge source.

3. RETRIEVAL-AUGMENTED GENERATION

Retrieval-Augmented Generation, commonly called RAG, is an AI technique
that combines information retrieval with language generation.


Source 2 | Similarity: 0.3937
StudyMate uses five external APIs to extend its research capabil

In [29]:
# ============================================================
# CELL 26 — STUDYMATE STREAMLIT APPLICATION
# ============================================================

app_code = r'''
import streamlit as st
import requests
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# ------------------------------------------------------------
# PAGE CONFIG
# ------------------------------------------------------------

st.set_page_config(
    page_title="StudyMate - Academic RAG Assistant",
    page_icon="🎓",
    layout="wide"
)


# ------------------------------------------------------------
# CUSTOM CSS
# ------------------------------------------------------------

st.markdown("""
<style>

.main-title {
    font-size: 42px;
    font-weight: 700;
    text-align: center;
    margin-bottom: 5px;
}

.subtitle {
    text-align: center;
    color: #666;
    font-size: 18px;
    margin-bottom: 30px;
}

.source-box {
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #ddd;
    margin-bottom: 12px;
}

.api-box {
    padding: 12px;
    border-radius: 10px;
    border: 1px solid #ddd;
    margin-bottom: 10px;
}

</style>
""", unsafe_allow_html=True)


# ------------------------------------------------------------
# LOAD RAG MODELS
# ------------------------------------------------------------

@st.cache_resource
def load_embedding_model():

    return SentenceTransformer(
        "all-MiniLM-L6-v2"
    )


@st.cache_resource
def load_llm():

    tokenizer = AutoTokenizer.from_pretrained(
        "google/flan-t5-small"
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(
        "google/flan-t5-small"
    )

    return tokenizer, model


embedding_model = load_embedding_model()

tokenizer, llm_model = load_llm()


# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

@st.cache_data
def load_dataset():

    with open(
        "/content/academic_notes.txt",
        "r",
        encoding="utf-8"
    ) as f:

        return f.read()


text = load_dataset()


# ------------------------------------------------------------
# CREATE CHUNKS
# ------------------------------------------------------------

@st.cache_data
def create_chunks(text):

    paragraphs = [
        p.strip()
        for p in text.split("\n\n")
        if p.strip()
    ]

    chunks = []

    current = ""

    for paragraph in paragraphs:

        if len(current) + len(paragraph) <= 900:

            current += paragraph + "\n\n"

        else:

            if current.strip():
                chunks.append(current.strip())

            current = paragraph + "\n\n"

    if current.strip():
        chunks.append(current.strip())

    return chunks


chunks = create_chunks(text)


# ------------------------------------------------------------
# CREATE FAISS INDEX
# ------------------------------------------------------------

@st.cache_resource
def create_index(chunks):

    embeddings = embedding_model.encode(
        chunks,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    index = faiss.IndexFlatIP(
        embeddings.shape[1]
    )

    index.add(embeddings)

    return index


index = create_index(chunks)


# ------------------------------------------------------------
# RAG RETRIEVER
# ------------------------------------------------------------

def retrieve_documents(
    question,
    top_k=3
):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        if idx >= 0:

            results.append({
                "score": float(score),
                "text": chunks[idx]
            })

    return results


# ------------------------------------------------------------
# RAG GENERATION
# ------------------------------------------------------------

def ask_rag(question):

    results = retrieve_documents(
        question,
        3
    )

    context = "\n\n".join(
        result["text"]
        for result in results
    )

    prompt = f"""
You are StudyMate, an academic assistant.

Answer the question using only the provided context.

Give a clear and concise answer.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=120,
        num_beams=4,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, results


# ------------------------------------------------------------
# WIKIPEDIA API
# ------------------------------------------------------------

def wikipedia_api(topic):

    url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + topic.replace(" ", "_")

    headers = {
        "User-Agent": "StudyMateAcademicAssistant/1.0"
    }

    try:

        response = requests.get(
            url,
            headers=headers,
            timeout=15
        )

        if response.status_code == 200:

            data = response.json()

            return {
                "title": data.get("title"),
                "summary": data.get("extract"),
                "url": data.get("content_urls", {})
                    .get("desktop", {})
                    .get("page")
            }

    except Exception:
        pass

    return None


# ------------------------------------------------------------
# ARXIV API
# ------------------------------------------------------------

def arxiv_api(query):

    try:

        url = "https://export.arxiv.org/api/query"

        params = {
            "search_query": f"all:{query}",
            "start": 0,
            "max_results": 3
        }

        response = requests.get(
            url,
            params=params,
            timeout=20
        )

        if response.status_code == 200:

            import feedparser

            feed = feedparser.parse(
                response.text
            )

            return [
                {
                    "title": e.get("title", "").strip(),
                    "url": e.get("link", "")
                }
                for e in feed.entries
            ]

    except Exception:
        pass

    return []


# ------------------------------------------------------------
# OPENALEX API
# ------------------------------------------------------------

def openalex_api(query):

    try:

        response = requests.get(
            "https://api.openalex.org/works",
            params={
                "search": query,
                "per-page": 3
            },
            timeout=20
        )

        if response.status_code == 200:

            data = response.json()

            return [
                {
                    "title": x.get("title"),
                    "year": x.get("publication_year"),
                    "doi": x.get("doi")
                }
                for x in data.get("results", [])
            ]

    except Exception:
        pass

    return []


# ------------------------------------------------------------
# CROSSREF API
# ------------------------------------------------------------

def crossref_api(query):

    try:

        response = requests.get(
            "https://api.crossref.org/works",
            params={
                "query": query,
                "rows": 3
            },
            timeout=20
        )

        if response.status_code == 200:

            data = response.json()

            return [
                {
                    "title": x.get("title", [""])[0],
                    "doi": x.get("DOI"),
                    "publisher": x.get("publisher")
                }
                for x in data["message"]["items"]
            ]

    except Exception:
        pass

    return []


# ------------------------------------------------------------
# OPEN LIBRARY API
# ------------------------------------------------------------

def openlibrary_api(query):

    try:

        response = requests.get(
            "https://openlibrary.org/search.json",
            params={
                "q": query,
                "limit": 3
            },
            timeout=20
        )

        if response.status_code == 200:

            data = response.json()

            return [
                {
                    "title": x.get("title"),
                    "author": x.get(
                        "author_name",
                        ["Unknown"]
                    )[0]
                }
                for x in data.get("docs", [])
            ]

    except Exception:
        pass

    return []


# ============================================================
# UI
# ============================================================

st.markdown(
    '<div class="main-title">🎓 StudyMate</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">'
    'Academic Retrieval-Augmented Generation Assistant'
    '</div>',
    unsafe_allow_html=True
)


# ------------------------------------------------------------
# SIDEBAR
# ------------------------------------------------------------

with st.sidebar:

    st.header("📚 StudyMate")

    st.write(
        "AI-powered academic assistant "
        "using Retrieval-Augmented Generation."
    )

    st.divider()

    st.subheader("🔧 Technology")

    st.write("• Sentence Transformers")
    st.write("• FAISS")
    st.write("• FLAN-T5")
    st.write("• Streamlit")
    st.write("• FastAPI")

    st.divider()

    st.subheader("🌐 External APIs")

    st.write("✅ Wikipedia")
    st.write("✅ arXiv")
    st.write("✅ OpenAlex")
    st.write("✅ Crossref")
    st.write("✅ Open Library")


# ------------------------------------------------------------
# CHAT HISTORY
# ------------------------------------------------------------

if "history" not in st.session_state:

    st.session_state.history = []


# ------------------------------------------------------------
# QUESTION INPUT
# ------------------------------------------------------------

question = st.text_input(
    "💬 Enter your academic question",
    placeholder="Example: What is Retrieval-Augmented Generation?"
)


if st.button(
    "🤖 Ask StudyMate",
    use_container_width=True
):

    if not question.strip():

        st.warning(
            "Please enter a question."
        )

    else:

        with st.spinner(
            "Searching knowledge base and generating answer..."
        ):

            try:

                answer, sources = ask_rag(
                    question
                )

                st.session_state.history.append({
                    "question": question,
                    "answer": answer
                })

                st.success("Answer generated!")

                # ANSWER

                st.subheader("🤖 Answer")

                st.write(answer)


                # SOURCES

                st.subheader(
                    "📖 Retrieved Sources"
                )

                for i, source in enumerate(
                    sources,
                    1
                ):

                    with st.expander(
                        f"Source {i} — "
                        f"Similarity: "
                        f"{source['score']:.4f}"
                    ):

                        st.write(
                            source["text"]
                        )


                # EXTERNAL RESEARCH

                st.subheader(
                    "🌐 External Research"
                )

                tabs = st.tabs([
                    "Wikipedia",
                    "arXiv",
                    "OpenAlex",
                    "Crossref",
                    "Open Library"
                ])


                # Wikipedia

                with tabs[0]:

                    result = wikipedia_api(
                        question
                    )

                    if result:

                        st.write(
                            "**" +
                            str(result["title"]) +
                            "**"
                        )

                        st.write(
                            result["summary"]
                        )

                        if result["url"]:

                            st.markdown(
                                f"[Open Wikipedia page]"
                                f"({result['url']})"
                            )

                    else:

                        st.info(
                            "No Wikipedia result."
                        )


                # arXiv

                with tabs[1]:

                    results = arxiv_api(
                        question
                    )

                    for paper in results:

                        st.markdown(
                            f"**{paper['title']}**"
                        )

                        st.markdown(
                            f"[View paper]"
                            f"({paper['url']})"
                        )


                # OpenAlex

                with tabs[2]:

                    results = openalex_api(
                        question
                    )

                    for item in results:

                        st.write(
                            item["title"]
                        )

                        st.write(
                            "Year:",
                            item["year"]
                        )

                        if item["doi"]:

                            st.write(
                                "DOI:",
                                item["doi"]
                            )


                # Crossref

                with tabs[3]:

                    results = crossref_api(
                        question
                    )

                    for item in results:

                        st.write(
                            item["title"]
                        )

                        st.write(
                            "Publisher:",
                            item["publisher"]
                        )

                        st.write(
                            "DOI:",
                            item["doi"]
                        )


                # Open Library

                with tabs[4]:

                    results = openlibrary_api(
                        question
                    )

                    for book in results:

                        st.write(
                            "📚",
                            book["title"]
                        )

                        st.write(
                            "Author:",
                            book["author"]
                        )


                # DOWNLOAD

                download_text = (
                    "StudyMate Answer\n\n"
                    f"Question:\n{question}\n\n"
                    f"Answer:\n{answer}\n"
                )

                st.download_button(
                    "📥 Download Answer",
                    download_text,
                    file_name="studymate_answer.txt"
                )


            except Exception as e:

                st.error(
                    "An error occurred: "
                    + str(e)
                )


# ------------------------------------------------------------
# CONVERSATION HISTORY
# ------------------------------------------------------------

if st.session_state.history:

    st.divider()

    st.subheader(
        "💬 Conversation History"
    )

    for item in reversed(
        st.session_state.history
    ):

        st.markdown(
            f"**You:** {item['question']}"
        )

        st.markdown(
            f"**StudyMate:** {item['answer']}"
        )
'''

with open(
    "/content/app.py",
    "w",
    encoding="utf-8"
) as f:

    f.write(app_code)

print("✅ Streamlit app.py created")
print("📄 Location: /content/app.py")

✅ Streamlit app.py created
📄 Location: /content/app.py


In [30]:
# CELL 27 — CHECK STREAMLIT FILE

import os

app_path = "/content/app.py"

print("Checking Streamlit application...")
print("=" * 60)

if os.path.exists(app_path):

    print("✅ app.py exists")

    size = os.path.getsize(app_path)

    print("📄 File size:", size, "bytes")

    with open(
        app_path,
        "r",
        encoding="utf-8"
    ) as f:

        content = f.read()

    print("📏 Lines:", len(content.splitlines()))

    print("\n✅ Required components:")

    checks = {
        "Streamlit": "import streamlit",
        "FAISS": "import faiss",
        "Sentence Transformers": "SentenceTransformer",
        "FLAN-T5": "flan-t5-small",
        "Wikipedia API": "wikipedia_api",
        "arXiv API": "arxiv_api",
        "OpenAlex API": "openalex_api",
        "Crossref API": "crossref_api",
        "Open Library API": "openlibrary_api",
        "RAG": "ask_rag"
    }

    for name, keyword in checks.items():

        if keyword in content:
            print(f"   ✅ {name}")
        else:
            print(f"   ❌ {name}")

else:

    print("❌ app.py was not found")

Checking Streamlit application...
✅ app.py exists
📄 File size: 16185 bytes
📏 Lines: 775

✅ Required components:
   ✅ Streamlit
   ✅ FAISS
   ✅ Sentence Transformers
   ✅ FLAN-T5
   ✅ Wikipedia API
   ✅ arXiv API
   ✅ OpenAlex API
   ✅ Crossref API
   ✅ Open Library API
   ✅ RAG


In [31]:
# CELL 28 — START STREAMLIT

import subprocess
import time
import os

# Stop any old Streamlit process
os.system("pkill -f 'streamlit run' 2>/dev/null")

time.sleep(2)

# Start Streamlit
process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true",
        "--server.enableCORS",
        "false",
        "--server.enableXsrfProtection",
        "false"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print("🚀 StudyMate Streamlit server started")
print("📌 Port: 8501")
print("⏳ Creating Colab access link...")

🚀 StudyMate Streamlit server started
📌 Port: 8501
⏳ Creating Colab access link...


In [32]:
# CELL 29 — CREATE COLAB STREAMLIT LINK

from google.colab.output import eval_js

try:

    url = eval_js(
        "google.colab.kernel.proxyPort(8501)"
    )

    print("🎓 STUDYMATE IS READY!")
    print("=" * 60)
    print("Open this link:")
    print(url)

except Exception as e:

    print("❌ Could not create Colab link")
    print(e)

🎓 STUDYMATE IS READY!
Open this link:
https://8501-m-s-kkb-usc1a1-3lth3b9jl6376-a.us-central1-1.prod.colab.dev


In [33]:
# CELL 30 — CREATE SUBMISSION ZIP

import os
import shutil

project_dir = "/content/StudyMate_RAG_Project"

# Create project folder
os.makedirs(project_dir, exist_ok=True)

# Copy application
shutil.copy(
    "/content/app.py",
    project_dir + "/app.py"
)

# Copy dataset if it exists
dataset_path = "/content/academic_notes.txt"

if os.path.exists(dataset_path):
    shutil.copy(
        dataset_path,
        project_dir + "/academic_notes.txt"
)

# Create requirements.txt
requirements = """streamlit
sentence-transformers
transformers
torch
faiss-cpu
requests
feedparser
fastapi
uvicorn
nest-asyncio
"""

with open(
    project_dir + "/requirements.txt",
    "w"
) as f:
    f.write(requirements)

# Create README
readme = """# StudyMate - Academic RAG Assistant

## Project Description

StudyMate is an AI-powered Academic Retrieval-Augmented Generation
(RAG) application.

It retrieves relevant information from an academic dataset using
FAISS and Sentence Transformers and generates answers using FLAN-T5.

## Technologies

- Python
- Streamlit
- FastAPI
- FAISS
- Sentence Transformers
- FLAN-T5
- Retrieval-Augmented Generation

## External APIs

1. Wikipedia
2. arXiv
3. OpenAlex
4. Crossref
5. Open Library

## Run Locally

Install dependencies:

pip install -r requirements.txt

Run Streamlit:

streamlit run app.py

## RAG Pipeline

Dataset
-> Text Processing
-> Chunking
-> Embeddings
-> FAISS
-> Retriever
-> Prompt
-> FLAN-T5
-> Answer

## Features

- Academic question answering
- RAG-based retrieval
- Source document display
- External research APIs
- Conversation history
- Download answer
- FastAPI endpoint
"""

with open(
    project_dir + "/README.md",
    "w"
) as f:
    f.write(readme)

# Create ZIP
zip_path = shutil.make_archive(
    "/content/StudyMate_RAG_Project",
    "zip",
    project_dir
)

print("✅ Project ZIP created!")
print("📦", zip_path)

print("\nFiles included:")

for file in os.listdir(project_dir):
    print(" •", file)

✅ Project ZIP created!
📦 /content/StudyMate_RAG_Project.zip

Files included:
 • academic_notes.txt
 • requirements.txt
 • README.md
 • app.py


In [34]:
# CELL 31 — PREPARE FINAL GITHUB PROJECT

import os
import shutil

project_dir = "/content/StudyMate-RAG-Academic-Assistant"

# Remove old folder if it exists
if os.path.exists(project_dir):
    shutil.rmtree(project_dir)

# Create folders
os.makedirs(project_dir)
os.makedirs(project_dir + "/screenshots")

# --------------------------------------------------
# Copy application files
# --------------------------------------------------

if os.path.exists("/content/app.py"):
    shutil.copy(
        "/content/app.py",
        project_dir + "/app.py"
    )

if os.path.exists("/content/academic_notes.txt"):
    shutil.copy(
        "/content/academic_notes.txt",
        project_dir + "/academic_notes.txt"
    )

# --------------------------------------------------
# requirements.txt
# --------------------------------------------------

requirements = """streamlit
sentence-transformers
transformers
torch
faiss-cpu
requests
feedparser
fastapi
uvicorn
nest-asyncio
numpy
"""

with open(
    project_dir + "/requirements.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(requirements)

# --------------------------------------------------
# README.md
# --------------------------------------------------

readme = """# 🎓 StudyMate – Academic RAG Assistant

## Student Information

**Student Name:** Arti
**Roll Number:** MC2505

---

## Project Title

StudyMate – Academic Retrieval-Augmented Generation Assistant

---

## Project Description

StudyMate is an AI-powered academic assistant based on
Retrieval-Augmented Generation (RAG).

The system retrieves relevant information from an academic
knowledge base and uses FLAN-T5 to generate an answer.

---

## Technologies Used

- Python
- Streamlit
- FastAPI
- FAISS
- Sentence Transformers
- FLAN-T5
- Requests
- NumPy

---

## RAG Pipeline

Dataset

↓

Text Preprocessing

↓

Text Chunking

↓

Sentence Transformer Embeddings

↓

FAISS Vector Search

↓

Relevant Context Retrieval

↓

Prompt Engineering

↓

FLAN-T5

↓

Generated Answer

---

## Embedding Model

all-MiniLM-L6-v2

The embedding model converts text into numerical vectors
for semantic similarity search.

---

## Vector Database

FAISS (Facebook AI Similarity Search)

FAISS is used to efficiently search the document embeddings.

---

## Language Model

FLAN-T5-small

The model generates answers using the retrieved context.

---

## External APIs

StudyMate integrates five external APIs:

1. Wikipedia API
2. arXiv API
3. OpenAlex API
4. Crossref API
5. Open Library API

---

## Own REST API

StudyMate also provides a FastAPI endpoint:

POST /ask

Example request:

{
    "question": "What is Retrieval-Augmented Generation?",
    "top_k": 3
}

---

## Features

- Academic question answering
- Retrieval-Augmented Generation
- FAISS similarity search
- Source document retrieval
- Five external research APIs
- FastAPI REST API
- Streamlit interface
- Conversation history
- Download answer functionality

---

## Project Structure

StudyMate-RAG-Academic-Assistant/

├── app.py

├── academic_notes.txt

├── requirements.txt

├── README.md

└── screenshots/

---

## Installation

Install the required packages:

pip install -r requirements.txt

---

## Run the Application

streamlit run app.py

---

## Deployment

The application is intended to be deployed using
Streamlit Community Cloud.

---

## Author

Student Name: Arti

Roll Number: MC2505
"""

with open(
    project_dir + "/README.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(readme)

# --------------------------------------------------
# .gitignore
# --------------------------------------------------

gitignore = """__pycache__/
*.pyc
.ipynb_checkpoints/
.env
.venv/
venv/
"""

with open(
    project_dir + "/.gitignore",
    "w",
    encoding="utf-8"
) as f:
    f.write(gitignore)

# --------------------------------------------------
# Show project structure
# --------------------------------------------------

print("=" * 60)
print("🎓 STUDYMATE GITHUB PROJECT")
print("=" * 60)

for root, dirs, files in os.walk(project_dir):

    level = root.replace(project_dir, "").count(os.sep)

    indent = "    " * level

    print(indent + os.path.basename(root) + "/")

    for file in files:
        print(
            indent + "    " + file
        )

print("\n✅ GitHub-ready project created!")
print("📁", project_dir)

🎓 STUDYMATE GITHUB PROJECT
StudyMate-RAG-Academic-Assistant/
    academic_notes.txt
    requirements.txt
    README.md
    app.py
    .gitignore
    screenshots/

✅ GitHub-ready project created!
📁 /content/StudyMate-RAG-Academic-Assistant
